# Homography test

## Imports

In [1]:
from sources.baseline_detection import BaselineDetection
from sources.scale import Scale
import pandas as pd
import numpy as np

## Description

- Define start-point and end-point of the movement
- Initialize the BaselineDetection (homography componenent)
- Calculate distance between the two points
- True: Report the distance in the distance file of the video; False: Rework the homography system implementation

### Define points

In [ ]:
# consts
frame_points_path = "../data/data/eval_points.json"
schema_points_path = "../data/data/points_cropped_schema.json"

In [ ]:
def retrieve_data(coordinates_path):
    # load the data into a pandas DataFrame
    df = pd.read_csv(coordinates_path)

    # define last frame number
    max_frame = df["frame"].max()

    # first position of the movement
    start_pt_1 = df[(df["frame"] == 0) & (df["id"] == 1)]
    start_pt_2 = df[(df["frame"] == 0) & (df["id"] == 2)]

    # last position of the movement
    end_pt_1 = df[(df["frame"] == max_frame) & (df["id"] == 1)]
    end_pt_2 = df[(df["frame"] == max_frame) & (df["id"] == 2)]

    return start_pt_1, start_pt_2, end_pt_1, end_pt_2

In [10]:
# initialize BaselineDetection component for the homography
bd = BaselineDetection(
    frame_points_path=frame_points_path,
    schema_points_path=schema_points_path
)

### Calculate distance between the two points

In [5]:
def apply_homography(pt, h_matrix):
    point = np.array([pt[0], pt[1], 1.0])
    transformed = h_matrix @ point

    return transformed[:2] / transformed[2]

In [6]:
def convert_bounding_box_to_point(bounding_box):
    x1 = bounding_box["x1"].values[0]
    x2 = bounding_box["x2"].values[0]
    x = x1 + ((x2 - x1) / 2)

    y1 = bounding_box["y1"].values[0]
    y2 = bounding_box["y2"].values[0]
    y = y1 + ((y2 - y1) / 2)

    return (x, y)

In [7]:
def calcul_distances(start_pt_1, start_pt_2 ,end_pt_1, end_pt_2, h):
    scale = Scale(schema_points_path)

    # convert bounding boxes to schema points
    new_start_pt_1 = apply_homography(convert_bounding_box_to_point(start_pt_1), h)
    new_start_pt_2 = apply_homography(convert_bounding_box_to_point(start_pt_2), h)
    new_end_pt_1 = apply_homography(convert_bounding_box_to_point(end_pt_1), h)
    new_end_pt_2 = apply_homography(convert_bounding_box_to_point(end_pt_2), h)

    # Distance ID - 1

    distance_1 = np.sqrt(
                    (new_end_pt_1[0] - new_start_pt_1[0]) ** 2 +
                    (new_end_pt_1[1] - new_start_pt_1[1]) ** 2
                    ) * scale.scale

    # Distance ID - 2
    distance_2 = np.sqrt(
                    (new_end_pt_2[0] - new_start_pt_2[0]) ** 2 +
                    (new_end_pt_2[1] - new_start_pt_2[1]) ** 2
                    ) * scale.scale


    print("Right value: 15.0")
    print(f"Player ID - 1: {distance_1}")
    print(f"Player ID - 2: {distance_2}")

In [8]:
h, inv_h = bd.calculate_homography()
start_pt_1, start_pt_2, end_pt_1, end_pt_2 = retrieve_data("../data/courtvision-dataset/eval_1_tracks.csv")
calcul_distances(start_pt_1, start_pt_2, end_pt_1, end_pt_2, h)

Right value: 15.0
Player ID - 1: 17.514095025216275
Player ID - 2: 16.23830429037151
